<a href="https://colab.research.google.com/github/Project-MANAS-Research-AI/Jansi_Bapodara_Research_AI/blob/main/IOI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install transformer-lens torch matplotlib seaborn

In [ ]:
pip uninstall torch torchvision torchaudio

In [ ]:
pip install torch torchvision torchaudio

In [ ]:
import torch
import matplotlib.pyplot as plt
import seaborn as sns

from transformer_lens import HookedTransformer

In [ ]:
model = HookedTransformer.from_pretrained("gpt2")

In [ ]:
prompt = "When Mary and John went to the store, John gave a drink to"
correct_answer = " Mary"
wrong_answer = " John"

In [ ]:
tokens_prompt = model.to_tokens(prompt)
tokens_correct = model.to_tokens(correct_answer)
tokens_wrong = model.to_tokens(wrong_answer)

In [ ]:
print(model.to_str_tokens(tokens_prompt))
print(model.to_str_tokens(tokens_correct))
print(model.to_str_tokens(tokens_wrong))

In [ ]:
logits = model(tokens_prompt)
print(logits.shape)

In [ ]:
last_token_logits = logits[0,-1,:]
prob = last_token_logits.softmax(dim = -1)
print(prob)

In [ ]:
mary_token = model.to_tokens(" Mary")

john_token = model.to_tokens(" John")
print(mary_token)
print(john_token)

In [ ]:
print("Mary probability:", prob[5335].item())

print("John probability:", prob[1757].item())

In [ ]:
corrupt_prompt = "When Alice and John went to the store, John gave a drink to"
corrupt_tokens = model.to_tokens(corrupt_prompt)
print(model.to_str_tokens(corrupt_tokens))

In [ ]:
clean_logits,clean_cache= model.run_with_cache(prompt)
clean_cache.keys()

In [ ]:
corrupt_logits,corrupt_cache= model.run_with_cache(corrupt_prompt)
corrupt_cache.keys()

In [ ]:

mary_index = mary_token[0, 1]
john_index = john_token[0, 1]


def logit_difference(logits):
    final_logits = logits[0, -1, :]

    mary_logit = final_logits[mary_index].item()
    john_logit = final_logits[john_index].item()

    return mary_logit - john_logit

print("Clean:", logit_difference(clean_logits))
print("Corrupt:", logit_difference(corrupt_logits))

In [ ]:
layer = 0
head = 0
def patch_head(z, hook):

    clean_z = clean_cache[hook.name]

    z[:,:,head,:] = clean_z[:,:,head,:]

    return z

In [ ]:
patched_logits = model.run_with_hooks(

    corrupt_tokens,

    fwd_hooks=[
        (
            f"blocks.{layer}.attn.hook_z",
            patch_head
        )
    ]
)

In [ ]:
print(
    "Corrupt:",
    logit_difference(corrupt_logits)
)


print(
    "Patched:",
    logit_difference(patched_logits)
)

In [ ]:
results = torch.zeros(
    model.cfg.n_layers,
    model.cfg.n_heads
)
for layer in range(model.cfg.n_layers):

    for head in range(model.cfg.n_heads):


        def patch(z, hook):

            clean_z = clean_cache[hook.name]

            z[:,:,head,:] = clean_z[:,:,head,:]

            return z



        patched_logits = model.run_with_hooks(

            corrupt_tokens,

            fwd_hooks=[
                (
                f"blocks.{layer}.attn.hook_z",
                patch
                )
            ]
        )


        results[layer,head] = logit_difference(
            patched_logits
        )

In [ ]:
print(results)
print(results.shape)

In [ ]:
plt.figure(figsize=(10,6))


sns.heatmap(
    results.numpy(),
    annot=False
)


plt.xlabel("Attention Head")

plt.ylabel("Layer")

plt.title(
    "GPT-2 Small IOI Head Importance"
)


plt.show()

In [ ]:
top_heads = torch.topk(
    results.flatten(),
    10
)


print(top_heads)
for value,index in zip(
    top_heads.values,
    top_heads.indices
):

    layer = index // model.cfg.n_heads

    head = index % model.cfg.n_heads

    print(
        "Layer:",
        layer,
        "Head:",
        head,
        "Score:",
        value.item()
    )

Sentence 2

In [ ]:
prompt = "While John and Mary were working at the library, John decided to give a book to"
correct_answer = " Mary"
wrong_answer = " John"


In [ ]:
tokens_prompt = model.to_tokens(prompt)
print(model.to_str_tokens(tokens_prompt))

In [ ]:
logits = model(tokens_prompt)
print(logits.shape)


last_token_logits = logits[0,-1,:]
prob = last_token_logits.softmax(dim = -1)
print(prob)


mary_token = model.to_tokens(" Mary")

john_token = model.to_tokens(" John")
print(mary_token)
print(john_token)


In [ ]:
print("Mary probability:", prob[mary_token[0,1]].item())

print("John probability:", prob[john_token[0,1]].item())


In [ ]:
corrupt_prompt = "While John and Alice were working at the library, David decided to give a book to"
corrupt_tokens = model.to_tokens(corrupt_prompt)
print(model.to_str_tokens(corrupt_tokens))


clean_logits,clean_cache= model.run_with_cache(prompt)
clean_cache.keys()
corrupt_logits,corrupt_cache= model.run_with_cache(corrupt_prompt)
corrupt_cache.keys()

In [ ]:
mary_index = mary_token[0, 1]
john_index = john_token[0, 1]


def logit_difference(logits):
    final_logits = logits[0, -1, :]

    mary_logit = final_logits[mary_index].item()
    john_logit = final_logits[john_index].item()

    return mary_logit - john_logit

print("Clean:", logit_difference(clean_logits))
print("Corrupt:", logit_difference(corrupt_logits))

In [ ]:
layer = 0
head = 0
def patch_head(z, hook):

    clean_z = clean_cache[hook.name]

    z[:,:,head,:] = clean_z[:,:,head,:]

    return z


patched_logits = model.run_with_hooks(

    corrupt_tokens,

    fwd_hooks=[
        (
            f"blocks.{layer}.attn.hook_z",
            patch_head
        )
    ]
)



print(
    "Corrupt:",
    logit_difference(corrupt_logits)
)


print(
    "Patched:",
    logit_difference(patched_logits)
)

In [ ]:
results = torch.zeros(
    model.cfg.n_layers,
    model.cfg.n_heads
)
for layer in range(model.cfg.n_layers):

    for head in range(model.cfg.n_heads):


        def patch(z, hook):

            clean_z = clean_cache[hook.name]

            z[:,:,head,:] = clean_z[:,:,head,:]

            return z



        patched_logits = model.run_with_hooks(

            corrupt_tokens,

            fwd_hooks=[
                (
                f"blocks.{layer}.attn.hook_z",
                patch
                )
            ]
        )


        results[layer,head] = logit_difference(
            patched_logits
        )




In [ ]:
plt.figure(figsize=(10,6))


sns.heatmap(
    results.numpy(),
    annot=False
)


plt.xlabel("Attention Head")

plt.ylabel("Layer")

plt.title(
    "GPT-2 Small IOI Head Importance"
)


plt.show()




In [ ]:

top_heads = torch.topk(
    results.flatten(),
    10
)


print(top_heads)
for value,index in zip(
    top_heads.values,
    top_heads.indices
):

    layer = index // model.cfg.n_heads

    head = index % model.cfg.n_heads

    print(
        "Layer:",
        layer,
        "Head:",
        head,
        "Score:",
        value.item()
    )


sentence 3

In [ ]:
prompt = "When Robert and Emily got a pineapple at the cafe, Robert decided to give it to"
correct_answer = " Emily"
wrong_answer = " Robert"

In [ ]:
tokens_prompt = model.to_tokens(prompt)
print(model.to_str_tokens(tokens_prompt))

In [ ]:
logits = model(tokens_prompt)
print(logits.shape)


last_token_logits = logits[0,-1,:]
prob = last_token_logits.softmax(dim = -1)
print(prob)


emily_token = model.to_tokens(" Emily")

robert_token = model.to_tokens(" Robert")
print(emily_token)
print(robert_token)

In [ ]:
print("Mary probability:", prob[emily_token[0,1]].item())

print("John probability:", prob[robert_token[0,1]].item())

In [ ]:
corrupt_prompt = "While John and Alice were working at the library, David decided to give a book to"
corrupt_tokens = model.to_tokens(corrupt_prompt)
print(model.to_str_tokens(corrupt_tokens))


clean_logits,clean_cache= model.run_with_cache(prompt)
clean_cache.keys()
corrupt_logits,corrupt_cache= model.run_with_cache(corrupt_prompt)
corrupt_cache.keys()

In [ ]:
emily_index = emily_token[0, 1]
robert_index = robert_token[0, 1]


def logit_difference(logits):
    final_logits = logits[0, -1, :]

    emily_logit = final_logits[emily_index].item()
    robert_logit = final_logits[robert_index].item()

    return emily_logit - robert_logit

print("Clean:", logit_difference(clean_logits))
print("Corrupt:", logit_difference(corrupt_logits))

In [ ]:
layer = 0
head = 0
def patch_head(z, hook):

    clean_z = clean_cache[hook.name]

    z[:,:,head,:] = clean_z[:,:,head,:]

    return z


patched_logits = model.run_with_hooks(

    corrupt_tokens,

    fwd_hooks=[
        (
            f"blocks.{layer}.attn.hook_z",
            patch_head
        )
    ]
)



print(
    "Corrupt:",
    logit_difference(corrupt_logits)
)


print(
    "Patched:",
    logit_difference(patched_logits)
)

In [ ]:
plt.figure(figsize=(10,6))


sns.heatmap(
    results.numpy(),
    annot=False
)


plt.xlabel("Attention Head")

plt.ylabel("Layer")

plt.title(
    "GPT-2 Small IOI Head Importance"
)


plt.show()




Layer 9, Head 9 stands out as the brightest, most critical spot. If you delete or break this one single head, the model’s ability to solve the sentence completely falls apart.Without Head 9, the model still understands the grammar of the sentence, but it forgets how to route the correct name to the very end.

Also we see in all the cases of logit difference that clean has always been greater than corrupt. Thus the model was confident while predicting the clean propmt but was less confident or unsure while prediciting the corrupt prompt.

We can also see that S inhibition. If there 2 same name in the sentence if senses that the next prediction should be the name that has only occured once

The experiment shows that corrupting the input reduces the model's preference for the correct indirect object, as seen from the drop in logit difference. This suggests that GPT-2 Small relies on specific internal pathways to transfer and process name information. Activation patching can be used to restore individual attention head outputs and identify the heads that contribute most to solving the IOI task.

The important attention heads with high recovery scores are likely involved in the IOI circuit, potentially acting as name mover or inhibition heads similar to those identified in the original paper.